# Focus

This notebooks focus is on checking the data and cleaning data from wikipedia

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make sure data and processed directories exist

In [2]:
RAW_DATA_PATH = Path("../../data/raw/wikipedia")
PROCESSED_DATA_PATH = Path("../../data/processed/source_data/wikipedia")

In [3]:
if not RAW_DATA_PATH.exists():
    print(f"Creating directory: {RAW_DATA_PATH}")
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {RAW_DATA_PATH}")
if not PROCESSED_DATA_PATH.exists():
    print(f"Creating directory: {PROCESSED_DATA_PATH}")
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {PROCESSED_DATA_PATH}")

Directory already exists: ..\..\data\raw\wikipedia
Directory already exists: ..\..\data\processed\wikipedia


# Load Raw Data

First, I loaded the file normally to inspect its structure and check what needs cleaning.

In [4]:
sp500 = pd.read_csv(RAW_DATA_PATH / "wikipedia_sp500_constituents.csv")
sp500.head()

,ticker,company_name,gics_sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [5]:
sp500.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ticker        503 non-null    str  
 1   company_name  503 non-null    str  
 2   gics_sector   503 non-null    str  
dtypes: str(3)
memory usage: 27.3 KB


# Check for Missing Values and Duplicates

In [6]:
sp500.isnull().sum()

ticker          0
company_name    0
gics_sector     0
dtype: int64

In [7]:
sp500.duplicated(subset='ticker').sum()

np.int64(0)

No missing values and no duplicate tickers, so the raw table is already in good shape structurally. The cleaning that actually matters here is fixing up the ticker symbol formatting so it can be used to merge with the price data later.

# Fix Ticker Symbol Formatting

Wikipedia writes some tickers with a period, like BRK.B, while Yahoo Finance and most other sources use a dash instead, like BRK-B. If this isn't fixed, any future merge between this table and price data will silently drop those rows.

In [8]:
dotted_tickers = sp500[sp500['ticker'].str.contains('\\.', regex=True)]
dotted_tickers

,ticker,company_name,gics_sector
61,BRK.B,Berkshire Hathaway,Financials
76,BF.B,Brown–Forman,Consumer Staples


This shows which tickers actually have a period in them, so we know exactly what's getting changed instead of blindly replacing everything.

In [9]:
sp500['ticker'] = sp500['ticker'].str.replace('.', '-', regex=False)

In [10]:
sp500[sp500['ticker'].str.contains('-', regex=False)]

,ticker,company_name,gics_sector
61,BRK-B,Berkshire Hathaway,Financials
76,BF-B,Brown–Forman,Consumer Staples


Confirmed the tickers that had a period now show up with a dash instead.

# Clean Whitespace and Text Formatting

In [11]:
sp500['ticker'] = sp500['ticker'].str.strip()
sp500['company_name'] = sp500['company_name'].str.strip()
sp500['gics_sector'] = sp500['gics_sector'].str.strip()

Stripping any extra whitespace just in case, this is a quick safety check since stray spaces can cause silent mismatches during a merge later.

# Inspect Cleaned Data

In [12]:
sp500.head()

,ticker,company_name,gics_sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [13]:
sp500['gics_sector'].nunique()

11

Confirms the number of unique sectors, which is a quick sanity check that the sector column is consistent and clean.

# Save Cleaned Data

In [14]:
sp500.to_csv(PROCESSED_DATA_PATH / "wikipedia_sp500_constituents_clean.csv", index=False)

# Final Summary

In this notebook it was focused on loading in the raw S&P 500 wikipedia table, checking for missing values and duplicates, fixing ticker symbol formatting so it can match up with price data later, cleaning up whitespace, and saving the processed dataset.

### Files Created
- `data/processed/source_data/wikipedia/wikipedia_sp500_constituents_clean.csv`